### Salary Prediction ML Model:
Using sklearn and decision trees

In [1]:
# 1. Standard libraries
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 2. Third-party packages
import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

In [2]:
# Directory artifacts where the pickled model joblib file will be stored
ART = Path("artifacts")
ART.mkdir(exist_ok=True)

In [3]:
data = pd.read_csv("./ds_salaries.csv")

In [4]:
data.columns

Index(['Unnamed: 0', 'work_year', 'experience_level', 'employment_type',
       'job_title', 'salary', 'salary_currency', 'salary_in_usd',
       'employee_residence', 'remote_ratio', 'company_location',
       'company_size'],
      dtype='str')

In [5]:
# no null values present
data.isnull().sum()

Unnamed: 0            0
work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64

In [6]:
# choosing the required / suitable features and target for salary prediction
df = data[["experience_level", "employment_type",
           "job_title", "company_location","company_size",
           "remote_ratio", "salary_in_usd"]]

In [7]:
X = df.drop("salary_in_usd", axis = 1)
y = df["salary_in_usd"]

In [8]:
X.head()

,experience_level,employment_type,job_title,company_location,company_size,remote_ratio
0,MI,FT,Data Scientist,DE,L,0
1,SE,FT,Machine Learning Scientist,JP,S,0
2,SE,FT,Big Data Engineer,GB,M,50
3,MI,FT,Product Data Analyst,HN,S,0
4,SE,FT,Machine Learning Engineer,US,L,50


In [9]:
categorical_columns = ["experience_level","employment_type","company_size","job_title", "company_location"]
numerical_columns = ["remote_ratio"]

In [10]:
all_categories = [
    df["experience_level"].unique().tolist(),
    df["employment_type"].unique().tolist(),
    df["company_size"].unique().tolist(),
    df["job_title"].unique().tolist(),
    df["company_location"].unique().tolist()
]

# Error when category is not in all_categories, handle_unknown
ohe = OneHotEncoder(categories=all_categories, handle_unknown="error")

In [11]:
categorical_pipe = Pipeline([
    ("ohe", ohe)
])

numerical_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])

pre_processing = ColumnTransformer([("cat", categorical_pipe, categorical_columns),
                                   ("num", numerical_pipe, numerical_columns)])

# max_depth:
# How many layers down the tree can grow
# min_samples_split:
# Minimum number of data points required in a node before the tree can split it into further branches
# min_samples_leaf:
# The minimum number of data points that must be present in a final "leaf" node
# Prevents the tree from creating overly specific tiny branches
pipe = Pipeline([("pre", pre_processing),
 ("model", DecisionTreeRegressor(max_depth=42, random_state=0, min_samples_split = 15, min_samples_leaf = 15))])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

# Evaluate
print("Mean Absolute Error (MAE): {:.2f}".format(mean_absolute_error(y_test, y_pred)))
print("Mean Squared Error (MSE): {:.2f}".format(mean_squared_error(y_test, y_pred)))
print("Root Mean Squared Error (RMSE): {:.2f}".format(np.sqrt(mean_squared_error(y_test, y_pred))))
print("R² Score: {:.2f}".format(r2_score(y_test, y_pred)))

Mean Absolute Error (MAE): 31658.11
Mean Squared Error (MSE): 1835383130.12
Root Mean Squared Error (RMSE): 42841.37
R² Score: 0.52


In [13]:
salary_actual = list(y_test[0:3])
salary_actual = [round(salary, 3) for salary in salary_actual]
salary_predicted = list(pipe.predict(X_test[0:3]))
salary_predicted = [round(salary, 3) for salary in salary_predicted]

print("Predicted vs acual salaries for the first 3 test samples")
print("--------------------------------------------------------")

print(f"(Actual) \t {salary_actual[0]} \t {salary_actual[1]} \t {salary_actual[2]}")
print(f"(Predicted) \t {salary_predicted[0]} \t {salary_predicted[1]} \t {salary_predicted[2]}")

Predicted vs acual salaries for the first 3 test samples
--------------------------------------------------------
(Actual) 	 140250 	 135000 	 100000
(Predicted) 	 148414.125 	 148414.125 	 137378.067


In [14]:
joblib.dump(pipe, ART / "pipeline.joblib")
print("Saved pipeline to artifacts/pipeline.joblib")

Saved pipeline to artifacts/pipeline.joblib
